In [ ]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/frontend/chunk_graph.js
import * as d3 from "https://esm.sh/d3@7";

export function render({ model, el }) {
  el.innerHTML = "";

  const container = d3.select(el)
    .append("div")
    .style("position", "relative")
    .style("width", "650px")
    .style("font-family", "sans-serif");

  const width = 650;
  const height = 420;

  const svg = container.append("svg")
    .attr("width", width)
    .attr("height", height)
    .style("background", "#f8fafc")
    .style("border", "1px solid #cbd5e1")
    .style("border-radius", "8px")
    .style("cursor", "grab");

  // Contenitore SVG scalabile e traslabile
  const g = svg.append("g");

  // Gestore dello Zoom (rotella) e Pan (drag sullo sfondo)
  const zoom = d3.zoom()
    .scaleExtent([0.2, 4]) // Zoom da 20% a 400%
    .on("zoom", (event) => {
      g.attr("transform", event.transform);
    });

  svg.call(zoom);

  const infoBox = container.append("div")
    .style("position", "absolute")
    .style("top", "12px")
    .style("right", "12px")
    .style("width", "220px")
    .style("padding", "10px")
    .style("background", "rgba(255, 255, 255, 0.95)")
    .style("border", "1px solid #cbd5e1")
    .style("border-radius", "6px")
    .style("font-size", "12px")
    .style("color", "#334155")
    .style("pointer-events", "none")
    .html("<b>📌 Info Chunk</b><br><span style='color:#94a3b8;'>Clicca un nodo per vederne il testo</span>");

  function draw() {
    const graph = model.get("graph_data");
    if (!graph || !graph.nodes || graph.nodes.length === 0) return;

    g.selectAll("*").remove();

    const nodes = graph.nodes.map(d => ({ ...d }));
    const links = graph.links ? graph.links.map(d => ({ ...d })) : [];

    const BASE_DISTANCE = 120;

    const simulation = d3.forceSimulation(nodes)
      .force("link", d3.forceLink(links)
        .id(d => d.id)
        .distance(d => BASE_DISTANCE * (d.distance_factor || 1.0))
      )
      .force("charge", d3.forceManyBody().strength(-180))
      .force("center", d3.forceCenter(width / 2, height / 2));

    const link = g.append("g")
      .selectAll("line")
      .data(links)
      .enter().append("line")
      .attr("stroke", "#94a3b8")
      .attr("stroke-width", 2);

    const linkText = g.append("g")
      .selectAll("text")
      .data(links)
      .enter().append("text")
      .attr("font-size", "11px")
      .attr("font-weight", "bold")
      .attr("fill", "#0284c7")
      .attr("text-anchor", "middle");

    const node = g.append("g")
      .selectAll("circle")
      .data(nodes)
      .enter().append("circle")
      .attr("r", 12)
      .attr("fill", "#6366f1")
      .attr("stroke", "#ffffff")
      .attr("stroke-width", 2)
      .style("cursor", "pointer");

    const label = g.append("g")
      .selectAll("text")
      .data(nodes)
      .enter().append("text")
      .text(d => d.id)
      .attr("font-size", "11px")
      .attr("dx", 15)
      .attr("dy", 4)
      .attr("fill", "#1e293b");

    simulation.on("end", () => {
      nodes.forEach(n => {
        n.fx = n.x;
        n.fy = n.y;
      });
    });

    const drag = d3.drag()
      .on("start", (event, d) => {
        nodes.forEach(n => {
          n.fx = n.x;
          n.fy = n.y;
        });
      })
      .on("drag", (event, d) => {
        d.fx = event.x;
        d.fy = event.y;
        d.x = event.x;
        d.y = event.y;
        updatePositions();
      })
      .on("end", (event, d) => {
        d.fx = event.x;
        d.fy = event.y;
        d.x = event.x;
        d.y = event.y;

        links.forEach(l => {
          const srcId = typeof l.source === 'object' ? l.source.id : l.source;
          const tgtId = typeof l.target === 'object' ? l.target.id : l.target;

          if (srcId === d.id || tgtId === d.id) {
            const dx = l.target.x - l.source.x;
            const dy = l.target.y - l.source.y;
            const currentDist = Math.sqrt(dx * dx + dy * dy);
            const factor = currentDist / BASE_DISTANCE;

            l.distance_factor = factor;

            model.set("pairwise_edit", {
              chunk_1: srcId,
              chunk_2: tgtId,
              distance_factor: factor
            });
            model.save_changes();
          }
        });

        updatePositions();
      });

    node.call(drag);

    node.on("click", (event, d) => {
      node.attr("fill", n => n.id === d.id ? "#ef4444" : "#6366f1");
      infoBox.html(`<b>🆔 ${d.id}</b><br><span style="color:#334155;">📖 ${d.text || "Nessun testo"}</span>`);
      model.set("selected_tag", { id: d.id, text: d.text || "" });
      model.save_changes();
    });

    function updatePositions() {
      link
        .attr("x1", d => d.source.x)
        .attr("y1", d => d.source.y)
        .attr("x2", d => d.target.x)
        .attr("y2", d => d.target.y);

      linkText
        .attr("x", d => (d.source.x + d.target.x) / 2)
        .attr("y", d => (d.source.y + d.target.y) / 2 - 5)
        .text(d => {
          const dx = d.target.x - d.source.x;
          const dy = d.target.y - d.source.y;
          const dist = Math.round(Math.sqrt(dx * dx + dy * dy));
          const factor = (dist / BASE_DISTANCE).toFixed(2);
          return `${dist}px (${factor}x)`;
        });

      node
        .attr("cx", d => d.x)
        .attr("cy", d => d.y);

      label
        .attr("x", d => d.x)
        .attr("y", d => d.y);
    }

    simulation.on("tick", updatePositions);
  }

  model.on("change:graph_data", draw);
  draw();
}
EOF

In [ ]:
### RAG SUGGESTED
"""

cat &lt;&lt; 'EOF' &gt; \~/tesi\_graphrag/src/frontend/chunk\_graph.js import \* as d3 from "https://esm.sh/d3@7"; export function render({ model, el }) { el.innerHTML = ""; function str(val) { return String(val); } // Riferimento per arrestare la simulazione fisica precedente let currentSimulation = null; // --- LAYOUT PRINCIPALE FLEXBOX (AFFIANCATO) --- const mainContainer = d3.select(el) .append("div") .style("display", "flex") .style("flex-direction", "row") .style("gap", "15px") .style("font-family", "system-ui, -apple-system, sans-serif") .style("color", "#1e293b"); // --- COLONNA SINISTRA: GRAFO D3.js --- const graphContainer = mainContainer.append("div") .style("position", "relative") .style("width", "580px") .style("height", "420px"); const width = 580; const height = 420; const svg = graphContainer.append("svg") .attr("width", width) .attr("height", height) .style("background", "#f8fafc") .style("border", "1px solid #cbd5e1") .style("border-radius", "8px") .style("cursor", "grab"); const g = svg.append("g"); // Zoom e Pan const zoom = d3.zoom() .scaleExtent([0.2, 4]) .on("zoom", (event) =&gt; { g.attr("transform", event.transform); }); svg.call(zoom); // --- COLONNA DESTRA: PANNELLO DI CONTROLLO UNIFICATO --- const panel = mainContainer.append("div") .style("width", "320px") .style("height", "420px") .style("box-sizing", "border-box") .style("padding", "12px") .style("background", "#ffffff") .style("border", "1px solid #cbd5e1") .style("border-radius", "8px") .style("display", "flex") .style("flex-direction", "column") .style("gap", "12px") .style("overflow-y", "auto"); // Sezione 1: Legenda Tag Globali const legendSection = panel.append("div") .style("padding-bottom", "10px") .style("border-bottom", "1px solid #e2e8f0"); legendSection.append("div") .style("font-weight", "bold") .style("font-size", "13px") .style("margin-bottom", "6px") .html("🏷️ Tag Globali Attivi"); const legendContainer = legendSection.append("div") .style("display", "flex") .style("flex-wrap", "wrap") .style("gap", "6px"); // Sezione 2: Dettaglio e Gestione Nodo Selezionato const detailSection = panel.append("div") .style("display", "flex") .style("flex-direction", "column") .style("gap", "8px"); detailSection.append("div") .style("font-weight", "bold") .style("font-size", "13px") .html("📌 Chunk Selezionato"); const detailContent = detailSection.append("div") .style("font-size", "12px") .style("color", "#64748b") .html("<i>Clicca un nodo nel grafo per vederne il testo e gestirne i Tag.</i>"); let selectedNodeId = null; let currentNodes = []; function updateLegend() { const globalTags = model.get("global\_tags") || {}; legendContainer.html(""); const tagNames = Object.keys(globalTags); if (tagNames.length === 0) { legendContainer.html("<span>Nessun tag assegnato.</span>"); return; } tagNames.forEach(tagName =&gt; { const info = globalTags[tagName]; const badge = legendContainer.append("span") .style("display", "inline-flex") .style("align-items", "center") .style("gap", "4px") .style("padding", "2px 8px") .style("border-radius", "12px") .style("font-size", "11px") .style("color", "#ffffff") .style("font-weight", "500") .style("background", info.color || "#6366f1"); badge.html(\`${tagName} <span>${info.count}</span>\`); }); } function getNodeColor(d, globalTags) { const userTags = d.user\_tags || d.tags || []; if (userTags.length &gt; 0) { const firstTag = userTags; // FIX: estrazione corretta della stringa del primo tag if (globalTags &amp;&amp; globalTags[firstTag] &amp;&amp; globalTags[firstTag].color) { return globalTags[firstTag].color; } } return "#6366f1"; // Colore di default } function renderDetailPanel() { const graph = model.get("graph\_data"); const globalTags = model.get("global\_tags") || {}; if (!selectedNodeId || !graph || !graph.nodes) { detailContent.html("<i>Clicca un nodo nel grafo per vederne il testo e gestirne i Tag.</i>"); return; } const nodeData = graph.nodes.find(n =&gt; str(n.id) === str(selectedNodeId)); if (!nodeData) { detailContent.html("<i>Nodo non trovato.</i>"); return; } detailContent.html(""); // ID detailContent.append("div") .style("font-weight", "bold") .style("color", "#0f172a") .style("margin-bottom", "4px") .text(\`🆔 ${nodeData.id}\`); // Anteprima testo detailContent.append("div") .style("max-height", "110px") .style("overflow-y", "auto") .style("padding", "6px 8px") .style("background", "#f1f5f9") .style("border-radius", "4px") .style("font-size", "11px") .style("margin-bottom", "8px") .text(nodeData.text || "Nessun testo associato."); // Tag correnti del nodo detailContent.append("div") .style("font-weight", "600") .style("font-size", "11px") .style("margin-bottom", "4px") .text("Tag Associati:"); const currentTagsContainer = detailContent.append("div") .style("display", "flex") .style("flex-wrap", "wrap") .style("gap", "4px") .style("margin-bottom", "10px"); const userTags = nodeData.user\_tags || nodeData.tags || []; if (userTags.length === 0) { currentTagsContainer.append("span") .style("font-size", "11px") .style("color", "#94a3b8") .text("Nessun tag."); } else { userTags.forEach(t =&gt; { const tagColor = (globalTags[t] &amp;&amp; globalTags[t].color) ? globalTags[t].color : "#6366f1"; const tagPill = currentTagsContainer.append("span") .style("display", "inline-flex") .style("align-items", "center") .style("gap", "4px") .style("padding", "2px 6px") .style("border-radius", "4px") .style("background", tagColor) .style("color", "#ffffff") .style("font-size", "11px"); tagPill.append("span").text(t); tagPill.append("span") .style("cursor", "pointer") .style("font-weight", "bold") .style("margin-left", "2px") .text("✕") .on("click", () =&gt; { model.set("tag\_action", { action: "remove", chunk\_id: nodeData.id, tag: t, timestamp: Date.now() }); model.save\_changes(); }); }); } // Campo Input e Pulsante per aggiungere nuovo Tag const addTagBox = detailContent.append("div") .style("display", "flex") .style("gap", "4px"); const inputTag = addTagBox.append("input") .attr("type", "text") .attr("placeholder", "Nuovo Tag...") .style("flex", "1") .style("padding", "4px 6px") .style("border", "1px solid #cbd5e1") .style("border-radius", "4px") .style("font-size", "11px"); const submitTag = () =&gt; { const val = inputTag.property("value").trim(); // FIX: verifica che il tag non sia vuoto e non sia già presente sul nodo if (val &amp;&amp; !userTags.includes(val)) { model.set("tag\_action", { action: "add", chunk\_id: nodeData.id, tag: val, timestamp: Date.now() }); model.save\_changes(); inputTag.property("value", ""); } }; inputTag.on("keydown", (event) =&gt; { if (event.key === "Enter") { submitTag(); } }); addTagBox.append("button") .text("+ Aggiungi") .style("padding", "4px 8px") .style("background", "#2563eb") .style("color", "#ffffff") .style("border", "none") .style("border-radius", "4px") .style("cursor", "pointer") .style("font-size", "11px") .on("click", submitTag); } function draw() { // FIX: arresta la simulazione fisica precedente se ancora attiva if (currentSimulation) { currentSimulation.stop(); } const graph = model.get("graph\_data"); const globalTags = model.get("global\_tags") || {}; updateLegend(); renderDetailPanel(); if (!graph || !graph.nodes || graph.nodes.length === 0) return; // Preservazione delle posizioni esistenti dei nodi al ricaricamento/ridisegno const posMap = {}; if (currentNodes) { currentNodes.forEach(n =&gt; { posMap[str(n.id)] = { x: n.x, y: n.y, fx: n.fx, fy: n.fy }; }); } const nodes = graph.nodes.map(d =&gt; { const existing = posMap[str(d.id)]; if (existing) { return { ...d, x: existing.x, y: existing.y, fx: existing.fx, fy: existing.fy }; } return { ...d }; }); currentNodes = nodes; const links = graph.links ? graph.links.map(d =&gt; ({ ...d })) : []; const BASE\_DISTANCE = graph.base\_distance || 120; g.selectAll("\*").remove(); const simulation = d3.forceSimulation(nodes) .force("link", d3.forceLink(links) .id(d =&gt; d.id) .distance(d =&gt; BASE\_DISTANCE \* (d.distance\_factor || 1.0)) ) .force("charge", d3.forceManyBody().strength(-180)) .force("center", d3.forceCenter(width / 2, height / 2)); currentSimulation = simulation; // Salva il riferimento alla simulazione attiva const link = g.append("g") .selectAll("line") .data(links) .enter().append("line") .attr("stroke", "#94a3b8") .attr("stroke-width", 2); const linkText = g.append("g") .selectAll("text") .data(links) .enter().append("text") .attr("font-size", "11px") .attr("font-weight", "bold") .attr("fill", "#0284c7") .attr("text-anchor", "middle"); const node = g.append("g") .selectAll("circle") .data(nodes) .enter().append("circle") .attr("r", 13) .attr("fill", d =&gt; getNodeColor(d, globalTags)) .attr("stroke", d =&gt; str(d.id) === str(selectedNodeId) ? "#000000" : "#ffffff") .attr("stroke-width", d =&gt; str(d.id) === str(selectedNodeId) ? 3 : 2) .style("cursor", "grab"); const label = g.append("g") .selectAll("text") .data(nodes) .enter().append("text") .text(d =&gt; d.id) .attr("font-size", "11px") .attr("dx", 16) .attr("dy", 4) .attr("fill", "#1e293b"); // Congela i nodi al termine del posizionamento iniziale simulation.on("end", () =&gt; { nodes.forEach(n =&gt; { n.fx = n.x; n.fy = n.y; }); }); // Drag &amp; Drop rigido con batching delle distanze const drag = d3.drag() .on("start", (event, d) =&gt; { nodes.forEach(n =&gt; { n.fx = n.x; n.fy = n.y; }); }) .on("drag", (event, d) =&gt; { d.fx = event.x; d.fy = event.y; d.x = event.x; d.y = event.y; updatePositions(); }) .on("end", (event, d) =&gt; { d.fx = event.x; d.fy = event.y; d.x = event.x; d.y = event.y; // Raccoglie in blocco (batch) le modifiche su tutti gli archi collegati al nodo mosso const batch = []; links.forEach(l =&gt; { const srcId = typeof l.source === 'object' ? l.source.id : l.source; const tgtId = typeof l.target === 'object' ? l.target.id : l.target; if (str(srcId) === str(d.id) || str(tgtId) === str(d.id)) { const dx = l.target.x - l.source.x; const dy = l.target.y - l.source.y; const currentDist = Math.hypot(dx, dy); const factor = currentDist / BASE\_DISTANCE; l.distance\_factor = factor; batch.push({ chunk\_1: srcId, chunk\_2: tgtId, distance\_factor: factor }); } }); // Invio atomico del batch a Python if (batch.length &gt; 0) { model.set("pairwise\_edits\_batch", batch); model.save\_changes(); } updatePositions(); }); node.call(drag); node.on("click", (event, d) =&gt; { selectedNodeId = d.id; // Evidenzia bordo nodo selezionato node.attr("stroke", n =&gt; str(n.id) === str(d.id) ? "#000000" : "#ffffff") .attr("stroke-width", n =&gt; str(n.id) === str(d.id) ? 3 : 2); renderDetailPanel(); model.set("selected\_tag", { id: d.id, text: d.text || "" }); model.save\_changes(); }); function updatePositions() { link .attr("x1", d =&gt; d.source.x) .attr("y1", d =&gt; d.source.y) .attr("x2", d =&gt; d.target.x) .attr("y2", d =&gt; d.target.y); linkText .attr("x", d =&gt; (d.source.x + d.target.x) / 2) .attr("y", d =&gt; (d.source.y + d.target.y) / 2 - 5) .text(d =&gt; { const dx = d.target.x - d.source.x; const dy = d.target.y - d.source.y; const dist = Math.round(Math.hypot(dx, dy)); const factor = (dist / BASE\_DISTANCE).toFixed(2); return \`${dist}px (${factor}x)\`; }); node .attr("cx", d =&gt; d.x) .attr("cy", d =&gt; d.y); label .attr("x", d =&gt; d.x) .attr("y", d =&gt; d.y); } simulation.on("tick", updatePositions); } const handleGlobalTagsChange = () =&gt; { updateLegend(); renderDetailPanel(); draw(); }; model.on("change:graph\_data", draw); model.on("change:global\_tags", handleGlobalTagsChange); draw(); // FIX: Funzione di cleanup allo smontaggio del widget in Anywidget return () =&gt; { if (currentSimulation) { currentSimulation.stop(); } model.off("change:graph\_data", draw); model.off("change:global\_tags", handleGlobalTagsChange); }; } EOF

"""